# 第16讲：修订同一个 Skill，保留旧能力

本讲设计题关注：已有单告警方法遇到多服务关联故障，怎样从新 Rollout 识别方法缺口，增补已有 Skill，并检查旧能力是否退步？多实例并发修订如何协调，仍是需要自行设计的思考题。

本练习按“新故障调查、主动后台复盘、查看同名补丁、新旧任务回放”操作，不把构造自检当成模型学习结果。


In [ ]:
from pathlib import Path
import ast
import json
import os
import signal
import subprocess
from IPython.display import Code, display

if "process" in globals() and process is not None:
    stop_owned()
HERE = Path.cwd().resolve()
if not (HERE / "native_workshop.py").exists():
    HERE = HERE / "examples/16-skill-incremental-patch"
assert (HERE / "native_workshop.py").is_file()
REPO = HERE.parents[1]
HERMES_SRC = Path(os.environ.get("HERMES_SRC", REPO / ".deps/hermes-agent")).resolve()
HERMES_PYTHON = Path(os.environ.get("HERMES_PYTHON", HERMES_SRC / ".venv/bin/python"))
assert HERMES_PYTHON.is_file(), "请先运行仓库 scripts/setup_deps.sh"
env = dict(os.environ, HERMES_SRC=str(HERMES_SRC), PYTHONDONTWRITEBYTECODE="1",
           PYTHONPATH=str(HERMES_SRC) + os.pathsep + os.environ.get("PYTHONPATH", ""))
process = None
live_result = None

def stop_owned():
    global process
    if process is not None and process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait(timeout=10)
    if process is not None and process.stdout is not None:
        process.stdout.close()
    process = None


## 先看本讲实际调用

前台使用新的 Session，核心调用为 `agent.run_conversation(user_message=..., system_message=..., conversation_history=[])`。后台显式调用：

```python
target, prompt = spawn_background_review_thread(
    parent, job["snapshot"], review_memory=False,
    review_skills=True, focus=scenario.FOCUS,
)
thread = threading.Thread(target=target, name="bg-review", daemon=True)
thread.start()
thread.join()
```

这是主动复盘，不是 Turn 收尾自动触发。下面直接展示本次要执行的函数及参数；观察器只记录返回，不改执行器或补写内容。


In [ ]:
source = (HERE / "native_workshop.py").read_text()
tree = ast.parse(source)
for node in tree.body:
    if isinstance(node, ast.FunctionDef) and node.name in ("foreground", "review_job"):
        display(Code(ast.get_source_segment(source, node), language="python"))


## 阅读旧方法与构造现场

旧 `oncall-alert-triage` 只覆盖单告警。事件 east 中两项告警共用故障组件，另一项同期告警有独立原因。工具返回当前指标、日志、依赖、调用链和探测结果，不能仅凭时间接近合并。

下面不请求模型。自检回执全部是明确构造的校验器输入，不依赖历史回放。


In [ ]:
import scenario
print("课程给定的旧方法：\n", scenario.SEED)
print("工具说明：\n", scenario.REFERENCE)
for script in ("native_workshop.py", "selfcheck.py"):
    checked = subprocess.run([str(HERMES_PYTHON), "-B", str(HERE / script)],
                             cwd=HERE, env=env, text=True, capture_output=True)
    print(checked.stdout)
    print(checked.stderr)
    checked.check_returncode()


## 新故障调查、复盘修订与新旧回放

设置当前进程的 `GLM_API_KEY`，把 `RUN_MODEL` 改为 `True`。默认 `glm-5.2`、官方接口，可用环境变量覆盖。调用会产生 API 费用。

前台先读旧 Skill 再调查 east。复盘依照原生提示和 `scenario.FOCUS` 先读后改，优先 `skill_manage(action="patch")` 修订同名主文件；引用按需修改并回读。脚本不提供新方法正文。

本格默认在独立教学库自动采用完整候选，记录 `teaching-auto`，不算人工审批。如需逐份确认，请在终端使用 `bash run.sh --live --no-auto-approve`，输入候选完整哈希；本格不会替你作人工确认。

随后创建两个空历史 Session：west 检验关联调查并保留未知，single 检验旧单告警路径。若无可靠修改，保留 `no-update` 与原生说明，不强行制造补丁。


In [ ]:
RUN_MODEL = os.environ.get("LESSON16_RUN_MODEL") == "1"  # 交互运行时可改为 True
live_result = None
if not RUN_MODEL:
    print("尚未运行模型；上面的现场检查不代表 Skill 修订成功。")
else:
    for name in ("GLM_API_KEY", "GLM_MODEL", "GLM_BASE_URL"):
        if name in os.environ:
            env[name] = os.environ[name]
    try:
        process = subprocess.Popen(
            [str(HERMES_PYTHON), "-B", str(HERE / "native_workshop.py"), "--live"],
            cwd=HERE, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, start_new_session=True)
        output, _ = process.communicate(timeout=900)
        print(output)
        if process.returncode:
            raise RuntimeError("本次模型链路未完成；查看本次输出与运行记录")
        for line in reversed(output.splitlines()):
            try:
                candidate = json.loads(line)
            except ValueError:
                continue
            if isinstance(candidate, dict) and candidate.get("path") == "native-live" and candidate.get("report"):
                live_result = candidate
                break
        assert live_result is not None, "没有本次运行的结果，不读取旧 latest 冒充"
    finally:
        stop_owned()


## 核对补丁、采用和新旧能力

下面只读取刚才进程返回的目录。逐条核对原生调用、成功回执、文件差异和最终回读，确认主文件保留单告警步骤，把多告警条件融入流程，而不是只追加教训。

两份未来任务记录应显示新 Session、空启动历史，以及查询前读取了采用版本的完整正文。程序检查调用和构造业务结果；是否形成通用方法、范围是否合适，仍需你阅读内容判断。


In [ ]:
import difflib
if live_result is None:
    print("尚无本次模型运行，不展示历史 Skill 充当本次结果。")
else:
    run = Path(live_result["report"]).parent
    review = json.loads((run / "review.json").read_text())
    print(review["result"]["final_response"])
    print(json.dumps(review["calls"], ensure_ascii=False, indent=2))
    changed = sorted(p for p in set(review["before"]) | set(review["after"])
                     if review["before"].get(p) != review["after"].get(p))
    for path in changed:
        print("\n文件：", path)
        print("".join(difflib.unified_diff(
            review["before"].get(path, "").splitlines(True),
            review["after"].get(path, "").splitlines(True), fromfile="before", tofile="after")))
        print("最终正文：\n", review["after"].get(path, "本次已移除"))
    for name in ("approval.json", "adoption.json"):
        if (run / name).exists():
            print(name, (run / name).read_text())
    for case in ("west", "single"):
        path = run / ("future-" + case + ".json")
        if path.exists():
            observation = json.loads(path.read_text())
            print(case, {k: observation[k] for k in (
                "session_id", "history_at_start", "loaded_adopted_content",
                "persisted_calls_match", "business_checks", "answer_checks", "answer_format")})
            print(json.dumps(observation["calls"], ensure_ascii=False, indent=2))
            print(observation["result"]["final_response"])
    print("本次结果：", live_result)


## 收尾与继续思考

停止本格启动的进程，关闭资源，保留独立 `runtime/native-*/` 记录；不停止其他运行。当前版本尚未完成模型端到端验证，源 Notebook 没有预置执行输出。

跨实例任务分布需由外层 Harness 汇总，本练习没有实现分布式任务汇总或修订协调。请思考：两个实例基于不同版本提出补丁时，怎样排队、检测冲突并重新审阅？两个回放通过以后，还需要哪些旧任务来发现行为退步？

固定 Hermes 只有 `_SKILL_REVIEW_PROMPT`，没有正文提及的另两个提示块变量；本练习不修改上游提示或造出同名变量。历史模型回放不用于本次验收。


In [ ]:
stop_owned()
